# Testing DINOv3 SAT with PCA Visualization

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision.transforms.functional import normalize
import torch.nn.functional as F
import rasterio

# ============================================================
# CONFIG
# ============================================================
DINO_REPO = r"C:\Users\osherr\dinov3"
DINO_CKPT = r"D:/Osher/depth2dem/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"

IMAGE_PATH = r"D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/RGB/000000000003.tif"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGENET_MEAN = (0.430, 0.411, 0.296)
IMAGENET_STD  = (0.213, 0.156, 0.143)

PATCH_SIZE = 16
STRIDE = 4
C = 1024  # feature dim

# ============================================================
# LOAD IMAGE
# ============================================================
def load_image(path):
    with rasterio.open(path) as src:
        img = src.read([1,2,3]).astype(np.float32) / 255.0
    return torch.from_numpy(img)

# ============================================================
# LOAD DINO
# ============================================================
def load_dino():
    model = torch.hub.load(
        DINO_REPO,
        "dinov3_vitl16",
        source="local",
        pretrained=False
    )

    ckpt = torch.load(DINO_CKPT, map_location="cpu")

    ckpt = {
        k.replace("module.", "").replace("teacher.backbone.", ""): v
        for k, v in ckpt.items()
    }

    msg = model.load_state_dict(ckpt, strict=False)

    print("\n===== LOAD REPORT =====")
    print("Missing:", msg.missing_keys)
    print("Unexpected:", msg.unexpected_keys)
    print("=======================\n")

    return model.to(DEVICE).eval()

# ============================================================
# PCA
# ============================================================
def compute_pca(tokens, H, W):
    X = tokens.cpu().numpy()
    X = X - X.mean(axis=0, keepdims=True)

    _, _, Vt = np.linalg.svd(X, full_matrices=False)
    X_pca = X @ Vt[:3].T

    img = X_pca.reshape(H, W, 3)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    return img

# ============================================================
# TRUE DENSE TOKENS (FIXED)
# ============================================================
def extract_dense_tokens(dino, rgb):

    p, stride = PATCH_SIZE, STRIDE

    rgb_pad = F.pad(rgb.unsqueeze(0), (p,p,p,p), mode="reflect")
    Hp_pad, Wp_pad = rgb_pad.shape[-2:]

    rgb_norm = normalize(rgb_pad, IMAGENET_MEAN, IMAGENET_STD).to(DEVICE)

    # dense grid
    Hp_out = Hp_pad // stride
    Wp_out = Wp_pad // stride

    feat_acc = torch.zeros((C, Hp_out, Wp_out), device=DEVICE)
    count_acc = torch.zeros((1, Hp_out, Wp_out), device=DEVICE)

    for dy in range(0, p, stride):
        for dx in range(0, p, stride):

            hc = ((Hp_pad - dy)//p)*p
            wc = ((Wp_pad - dx)//p)*p

            if hc <= 0 or wc <= 0:
                continue

            patch = rgb_norm[:,:,dy:dy+hc, dx:dx+wc]

            t = dino.forward_features(patch)["x_norm_patchtokens"].squeeze(0)
            t = t.reshape(hc//p, wc//p, C).permute(2,0,1)  # (C,H,W)

            feat_acc[:,
                     dy//stride : dy//stride + (hc//p)*(p//stride) : p//stride,
                     dx//stride : dx//stride + (wc//p)*(p//stride) : p//stride
            ] += t

            count_acc[:,
                      dy//stride : dy//stride + (hc//p)*(p//stride) : p//stride,
                      dx//stride : dx//stride + (wc//p)*(p//stride) : p//stride
            ] += 1

    feat_dense = feat_acc / (count_acc + 1e-8)

    # crop to original region
    H, W = rgb.shape[1:]
    Hp = H // stride
    Wp = W // stride

    offset = (p - (p//2)) // stride + 1

    feat_final = feat_dense[:, offset:offset+Hp, offset:offset+Wp]

    # reshape to (N, C)
    return feat_final.permute(1,2,0).reshape(-1, C), Hp, Wp

# ============================================================
# MAIN
# ============================================================
def main():

    rgb = load_image(IMAGE_PATH)
    H, W = rgb.shape[1:]

    dino = load_dino()

    with torch.no_grad():
        tokens, Hp, Wp = extract_dense_tokens(dino, rgb)

    print("Dense token shape:", tokens.shape)
    print(f"Dense grid: {Hp} x {Wp}")

    # PCA
    pca_img = compute_pca(tokens, Hp, Wp)

    # upscale
    pca_up = cv2.resize(pca_img, (W, H), interpolation=cv2.INTER_LINEAR)

    # visualize
    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.title("Input")
    plt.imshow(rgb.permute(1,2,0))
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title("DINO Dense PCA (correct)")
    plt.imshow(pca_up)
    plt.axis("off")

    plt.show()


if __name__ == "__main__":
    main()

# Relative Depth Model (DA-2) - third‑party model

In [2]:
cd C:\Users\osherr\MoGe #Local Path Cloned for GitHub

C:\Users\osherr\Depth-Anything-V2


In [ ]:
import cv2
import torch
# from moge.model.v1 import MoGeModel
from moge.model.v2 import MoGeModel # Let's try MoGe-2

device = torch.device("cuda")

# Load the model from huggingface hub (or load from local).
model = MoGeModel.from_pretrained("Ruicheng/moge-2-vitl").to(device)       

In [ ]:
import os
import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
import rasterio  # <--- Using rasterio instead of cv2

# --- CONFIGURATION ---
image_path = "D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/RGB/000000000003.tif"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 1. LOAD & PREPROCESS (Fixed)
# ============================================================
if not os.path.exists(image_path):
    raise FileNotFoundError(f"❌ File does not exist: {image_path}\nPlease check the path/filename.")

try:
    with rasterio.open(image_path) as src:
        # Rasterio reads as (Channels, Height, Width) -> (3, H, W)
        # We read the first 3 bands (RGB)
        input_data = src.read([1, 2, 3])
        
        # Prepare for Visualization: Transpose to (H, W, 3)
        input_image_rgb = input_data.transpose(1, 2, 0)

        # Prepare for Model: Normalize to [0, 1]
        # input_data is already (3, H, W), so we just convert to tensor
        input_tensor = torch.tensor(
            input_data / 255.0, 
            dtype=torch.float32, 
            device=device
        )
except Exception as e:
    raise RuntimeError(f"❌ Could not read TIFF file. Error: {e}")

# ============================================================
# 2. INFERENCE
# ============================================================
print(f"Running inference on {image_path}...")
with torch.no_grad():
    output = model.infer(input_tensor.unsqueeze(0)) # Add batch dim (1, 3, H, W) if model expects it, else remove unsqueeze

# Extract depth map
depth_map = output["depth"]
if isinstance(depth_map, torch.Tensor):
    depth_map = depth_map.squeeze().cpu().numpy() # Squ"eeze to remove batch dim if present

# Optional: Apply mask
if "mask" in output:
    mask = output["mask"]
    if isinstance(mask, torch.Tensor):
        mask = mask.squeeze().cpu().numpy()
    depth_map[mask < 0.5] = np.nan

# ============================================================
# 3. VISUALIZATION (Corrected for Aerial View)
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Plot RGB
ax[0].imshow(input_image_rgb)
ax[0].set_title("Input Image (RGB)")
ax[0].axis("off")

# Plot Depth
# 'inferno_r' reverses the map:
# Low values (Close/Roofs) -> Bright/Yellow
# High values (Far/Ground) -> Dark/Purple
im = ax[1].imshow(depth_map, cmap='inferno_r') 
ax[1].set_title("Predicted Depth Map (Brighter = Closer/Taller)")
ax[1].axis("off")

plt.colorbar(im, ax=ax[1], label="Distance from Camera (m)")
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
import matplotlib.pyplot as plt
from torchvision.transforms.functional import normalize
import cv2

# ============================================================
# CONFIG
# ============================================================
EXAMPLE_NAME = "000000000003.tif"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# Paths - Update these to your local environment
RGB_DIR    = r"D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/RGB/"
GT_DIR     = r"D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/GT_DSM/"
MASK_DIR   = r"D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/MASK/"
DA_DIR     = r"D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/RELATIVE_DEPTH/"
PRIOR_DIR  = r"D:/Osher/depth2dem/Prior2DSM_Final_predictions_and_code/FINAL_NOTEBOOKS/example_data/PRIORS/"

DINO_REPO = r"C:\Users\osherr\dinov3" #local Path
DINO_CKPT = r"D:/Osher/depth2dem/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth" #DINOv3 SAT Weights

DSM_BAND, MASK_BAND, DA_BAND, PRIOR_BAND = 6, 3, 1, 3
PATCH_SIZE = 16
STRIDE = 4

# Optimization Config
LORA_R = 8
LORA_ALPHA = 16.0
TTO_STEPS = 100
TTO_LR = 1e-3

IMAGENET_MEAN = (0.430, 0.411, 0.296)
IMAGENET_STD  = (0.213, 0.156, 0.143)

# ============================================================
# VIDEO CONFIG
# ============================================================
OUT_DIR = r"D:/Osher/depth2dem/lora_training_video/"
os.makedirs(OUT_DIR, exist_ok=True)

VIDEO_PATH = os.path.join(OUT_DIR, EXAMPLE_NAME.replace(".tif", "_real_process.mp4"))
FINAL_PNG_PATH = os.path.join(OUT_DIR, EXAMPLE_NAME.replace(".tif", "_final.png"))

SAVE_FRAME_EVERY = 1   # full dense inference every N steps
VIDEO_FPS = 4

# ============================================================
# HELPERS
# ============================================================
def normalize_01(arr, eps=1e-8):
    arr = np.asarray(arr, dtype=np.float32)
    valid = np.isfinite(arr)
    out = np.zeros_like(arr, dtype=np.float32)
    if not valid.any():
        return out
    vmin = float(arr[valid].min())
    vmax = float(arr[valid].max())
    out[valid] = (arr[valid] - vmin) / max(vmax - vmin, eps)
    return np.clip(out, 0.0, 1.0)

def tensor_to_np(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return x

# ============================================================
# MODELS: LoRA & MLP HEAD
# ============================================================
class MLPDecoder(nn.Module):
    def __init__(self, in_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 2)  # [Scale, Bias]
        )
        nn.init.zeros_(self.net[-1].weight)
        self.net[-1].bias.data = torch.tensor([1.0, 0.0])

    def forward(self, x):
        return self.net(x)

class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=16.0):
        super().__init__()
        self.base = base_linear

        # freeze original linear
        self.base.weight.requires_grad_(False)
        if getattr(self.base, "bias", None) is not None:
            self.base.bias.requires_grad_(False)

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r if r > 0 else 1.0

        self.A = nn.Linear(base_linear.in_features, r, bias=False)
        self.B = nn.Linear(r, base_linear.out_features, bias=False)

        nn.init.kaiming_uniform_(self.A.weight, a=np.sqrt(5))
        nn.init.zeros_(self.B.weight)

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

def inject_lora(model, r=8, alpha=16.0):
    for blk in model.modules():
        if hasattr(blk, "attn"):
            if hasattr(blk.attn, "qkv") and not isinstance(blk.attn.qkv, LoRALinear):
                blk.attn.qkv = LoRALinear(blk.attn.qkv, r, alpha)
            if hasattr(blk.attn, "proj") and not isinstance(blk.attn.proj, LoRALinear):
                blk.attn.proj = LoRALinear(blk.attn.proj, r, alpha)
    return model

def get_lora_params(model):
    params = []
    for module in model.modules():
        if isinstance(module, LoRALinear):
            params.extend(list(module.A.parameters()))
            params.extend(list(module.B.parameters()))
    return params

# ============================================================
# FULL DENSE INFERENCE
# ============================================================
@torch.no_grad()
def dense_predict_current(dino, mlp_head, rgb_cpu, rel_cpu, H, W, patch_size=16, stride=4):
    was_training_dino = dino.training
    was_training_head = mlp_head.training

    dino.eval()
    mlp_head.eval()

    p = patch_size
    rgb_pad = F.pad(rgb_cpu.unsqueeze(0), (p, p, p, p), mode="reflect")
    Hp_pad, Wp_pad = rgb_pad.shape[-2:]

    sb_acc = torch.zeros((2, Hp_pad // stride, Wp_pad // stride), device=DEVICE)
    cnt_acc = torch.zeros((1, Hp_pad // stride, Wp_pad // stride), device=DEVICE)

    rgb_norm = normalize(rgb_pad, IMAGENET_MEAN, IMAGENET_STD).to(DEVICE)

    for dy in range(0, p, stride):
        for dx in range(0, p, stride):
            hc = ((Hp_pad - dy) // p) * p
            wc = ((Wp_pad - dx) // p) * p
            if hc <= 0 or wc <= 0:
                continue

            patch = rgb_norm[:, :, dy:dy+hc, dx:dx+wc]
            t = dino.forward_features(patch)["x_norm_patchtokens"].squeeze(0)
            sb_local = mlp_head(t).t().reshape(2, hc // p, wc // p)

            sb_acc[:, dy//stride : dy//stride + (hc//p)*(p//stride) : p//stride,
                     dx//stride : dx//stride + (wc//p)*(p//stride) : p//stride] += sb_local

            cnt_acc[:, dy//stride : dy//stride + (hc//p)*(p//stride) : p//stride,
                       dx//stride : dx//stride + (wc//p)*(p//stride) : p//stride] += 1

    sb_dense = sb_acc / (cnt_acc + 1e-8)
    offset = (p - (p // 2)) // stride + 1
    sb_final = sb_dense[:, offset:offset + (H // stride), offset:offset + (W // stride)]

    sb_hr = F.interpolate(sb_final.unsqueeze(0), size=(H, W), mode="bilinear", align_corners=False).squeeze(0)
    s_hr, b_hr = sb_hr[0], sb_hr[1]

    pred = (s_hr * rel_cpu + b_hr).detach().cpu().numpy()

    if was_training_dino:
        dino.train()
    if was_training_head:
        mlp_head.train()

    return pred

# ============================================================
# VIDEO FRAME RENDER
# ============================================================
def make_video_frame(
    step,
    total_steps,
    rgb_cpu,
    rel_cpu,
    pred_fullres,
    gt_cpu,
    mask_cpu,
    loss_hist,
):
    rgb_np = rgb_cpu.permute(1, 2, 0).detach().cpu().numpy()
    rel_np = tensor_to_np(rel_cpu)
    gt_np = tensor_to_np(gt_cpu)
    mask_np = tensor_to_np(mask_cpu).astype(np.float32)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()

    axes[0].imshow(np.clip(rgb_np, 0, 1))
    axes[0].set_title("Input RGB")
    axes[0].axis("off")

    im1 = axes[1].imshow(rel_np, cmap="viridis")
    axes[1].set_title("Relative Depth Input")
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(pred_fullres, cmap="terrain")
    axes[2].set_title(f"Full-Resolution Prediction\nStep {step+1}/{total_steps}")
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    im3 = axes[3].imshow(gt_np, cmap="terrain")
    axes[3].set_title("Ground Truth DSM")
    axes[3].axis("off")
    plt.colorbar(im3, ax=axes[3], fraction=0.046)

    axes[4].imshow(mask_np, cmap="gray")
    axes[4].set_title("Target Mask")
    axes[4].axis("off")

    xs = np.arange(len(loss_hist))
    ys = np.array(loss_hist, dtype=np.float32)

    axes[5].plot(xs, ys, linewidth=2)
    if len(xs) > 0:
        axes[5].scatter([xs[0]], [ys[0]], s=45, label="start", zorder=3)
        axes[5].scatter([xs[-1]], [ys[-1]], s=45, label="current", zorder=3)
    axes[5].set_title("TTO Huber Loss")
    axes[5].set_xlabel("Step")
    axes[5].set_ylabel("Loss")
    axes[5].set_yscale("log")
    axes[5].grid(True, alpha=0.3)
    axes[5].legend()

    plt.tight_layout()

    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    frame = frame[:, :, :3].copy()
    plt.close(fig)
    return frame

# ============================================================
# DATA LOADING & PREP
# ============================================================
with rasterio.open(os.path.join(RGB_DIR, EXAMPLE_NAME)) as s:
    rgb_cpu = torch.from_numpy(s.read([1, 2, 3]).astype(np.float32) / 255.0)

with rasterio.open(os.path.join(GT_DIR, EXAMPLE_NAME)) as s:
    gt_cpu = torch.from_numpy(s.read(DSM_BAND).astype(np.float32))

with rasterio.open(os.path.join(MASK_DIR, EXAMPLE_NAME)) as s:
    mask_cpu = torch.from_numpy((s.read(MASK_BAND) > 0))

with rasterio.open(os.path.join(DA_DIR, EXAMPLE_NAME)) as s:
    rel_cpu = torch.from_numpy(s.read(DA_BAND).astype(np.float32)).to(DEVICE)

with rasterio.open(os.path.join(PRIOR_DIR, EXAMPLE_NAME)) as s:
    prior_raw = torch.from_numpy(s.read(PRIOR_BAND).astype(np.float32))

H, W = gt_cpu.shape
anchor_mask_cpu = (~mask_cpu) & torch.isfinite(prior_raw) & (prior_raw != 0)
anchor_mask = anchor_mask_cpu.to(DEVICE)
prior_gpu = prior_raw.to(DEVICE)

# ============================================================
# TRAINING (TTO)
# ============================================================
print("Initializing Models...")
dino = torch.hub.load(DINO_REPO, "dinov3_vitl16", source="local", pretrained=False)
sd = {
    k.replace("module.", "").replace("teacher.backbone.", ""): v
    for k, v in torch.load(DINO_CKPT, map_location="cpu").items()
}
dino.load_state_dict(sd, strict=False)

dino = inject_lora(dino, r=LORA_R, alpha=LORA_ALPHA).to(DEVICE).train()
mlp_head = MLPDecoder(in_dim=1024).to(DEVICE).train()

for p in dino.parameters():
    p.requires_grad_(False)
for p in get_lora_params(dino):
    p.requires_grad_(True)
for p in mlp_head.parameters():
    p.requires_grad_(True)

params = list(mlp_head.parameters()) + get_lora_params(dino)
opt = torch.optim.AdamW(params, lr=TTO_LR)

# Low-res TTO targets
rgb_tto = normalize(rgb_cpu.unsqueeze(0), IMAGENET_MEAN, IMAGENET_STD).to(DEVICE)
Hp, Wp = H // PATCH_SIZE, W // PATCH_SIZE

prior_p = F.interpolate(prior_gpu.view(1, 1, H, W), size=(Hp, Wp), mode="bilinear", align_corners=False).flatten()
rel_p   = F.interpolate(rel_cpu.view(1, 1, H, W), size=(Hp, Wp), mode="bilinear", align_corners=False).flatten()
mask_p  = F.interpolate(anchor_mask.float().view(1, 1, H, W), size=(Hp, Wp), mode="area").flatten() > 0.5

loss_hist = []

# prepare video writer based on first frame size later
video_writer = None

print(f"Starting TTO for {TTO_STEPS} steps...")
for step in range(TTO_STEPS):
    opt.zero_grad(set_to_none=True)

    tokens = dino.forward_features(rgb_tto)["x_norm_patchtokens"].squeeze(0)
    sb = mlp_head(tokens)
    s, b = sb[:, 0], sb[:, 1]

    pred_p = s * rel_p + b
    loss = F.huber_loss(pred_p[mask_p], prior_p[mask_p], delta=1.0)

    loss.backward()
    opt.step()

    loss_hist.append(loss.item())

    if step % 10 == 0 or step == TTO_STEPS - 1:
        print(f"Step {step:03d} | Loss: {loss.item():.6f}")

    # Save real full-resolution prediction frames
    if (step % SAVE_FRAME_EVERY == 0) or (step == TTO_STEPS - 1):
        pred_fullres = dense_predict_current(
            dino=dino,
            mlp_head=mlp_head,
            rgb_cpu=rgb_cpu,
            rel_cpu=rel_cpu,
            H=H,
            W=W,
            patch_size=PATCH_SIZE,
            stride=STRIDE
        )

        frame_rgb = make_video_frame(
            step=step,
            total_steps=TTO_STEPS,
            rgb_cpu=rgb_cpu,
            rel_cpu=rel_cpu,
            pred_fullres=pred_fullres,
            gt_cpu=gt_cpu,
            mask_cpu=mask_cpu,
            loss_hist=loss_hist
        )

        if video_writer is None:
            h_frame, w_frame = frame_rgb.shape[:2]
            video_writer = cv2.VideoWriter(
                VIDEO_PATH,
                cv2.VideoWriter_fourcc(*"mp4v"),
                VIDEO_FPS,
                (w_frame, h_frame)
            )

        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        video_writer.write(frame_bgr)

if video_writer is not None:
    video_writer.release()

print(f"Training video saved to: {VIDEO_PATH}")

# ============================================================
# FINAL FULL-RES INFERENCE
# ============================================================
print("Running final dense stride-4 prediction...")
final_dsm = dense_predict_current(
    dino=dino,
    mlp_head=mlp_head,
    rgb_cpu=rgb_cpu,
    rel_cpu=rel_cpu,
    H=H,
    W=W,
    patch_size=PATCH_SIZE,
    stride=STRIDE
)

# ============================================================
# FINAL VISUALIZATION
# ============================================================
plt.figure(figsize=(20, 10))

plt.subplot(2, 3, 1)
plt.title("Input RGB")
plt.imshow(rgb_cpu.permute(1, 2, 0).cpu().numpy())
plt.axis("off")

plt.subplot(2, 3, 2)
plt.title("Relative Depth Input")
plt.imshow(rel_cpu.detach().cpu().numpy(), cmap="viridis")
plt.colorbar(fraction=0.046)
plt.axis("off")

plt.subplot(2, 3, 3)
plt.title("Final LoRA DSM")
plt.imshow(final_dsm, cmap="terrain")
plt.colorbar(fraction=0.046)
plt.axis("off")

plt.subplot(2, 3, 4)
plt.title("Ground Truth DSM")
plt.imshow(gt_cpu.cpu().numpy(), cmap="terrain")
plt.colorbar(fraction=0.046)
plt.axis("off")

plt.subplot(2, 3, 5)
plt.title("Target Mask")
plt.imshow(mask_cpu.cpu().numpy(), cmap="gray")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.title("TTO Huber Loss")
plt.plot(loss_hist, linewidth=2)
plt.scatter([0], [loss_hist[0]], s=50, label="start", zorder=3)
plt.scatter([len(loss_hist)-1], [loss_hist[-1]], s=50, label="final", zorder=3)
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig(FINAL_PNG_PATH, dpi=200, bbox_inches="tight")
plt.show()

print(f"Final figure saved to: {FINAL_PNG_PATH}")

# ============================================================
# METRIC CHECK
# ============================================================
mask_np = mask_cpu.cpu().numpy()
mae = np.mean(np.abs(final_dsm[mask_np] - gt_cpu.cpu().numpy()[mask_np]))
print(f"Final MLP-TTO MAE: {mae:.4f}")